## 1️⃣ Install Dependencies

In [ ]:
# ============================================================
# Install Dependencies
# Supports fully open-source models — NO OpenAI key required
# ============================================================

# Core ML (PyTorch + HuggingFace) — for local model inference
!pip install -q transformers>=4.40.0 accelerate bitsandbytes sentencepiece
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q timm tiktoken einops

# FastAPI server + ngrok
!pip install -q fastapi uvicorn pyngrok nest-asyncio python-dotenv

# LangChain — open-source backends
!pip install -q langchain langchain-core langchain-community
!pip install -q langchain-groq          # Groq: free cloud (llama-3, mixtral)
!pip install -q langchain-ollama        # Ollama: local models
# !pip install -q langchain-openai      # OpenAI: optional paid backend

# Utility
!pip install -q Pillow psutil requests

# PowerPoint generation
!pip install -q python-pptx lxml
!pip install -q protobuf==3.20.3 --force-reinstall -q

print()
print("=" * 55)
print("✅ All dependencies installed!")
print()
print("Provider options (set LLM_PROVIDER below):")
print("  ollama       — local Ollama server (FREE, no key)")
print("  groq         — Groq cloud FREE tier (llama-3-70b)")
print("  huggingface  — HF Inference API (FREE tier)")
print("  transformers — local GPU model (this notebook)")
print("  openai       — GPT-4o (paid, optional)")
print("=" * 55)


## 2️⃣ Setup Ngrok

Get your free ngrok token from https://ngrok.com/

## 1.8️⃣ PowerPoint Generator Pro - Gamma/Canva Quality

Advanced layouts with professional themes and animations

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.text import PP_ALIGN
from pptx.enum.shapes import MSO_SHAPE
from pptx.dml.color import RGBColor
from datetime import datetime
from lxml import etree
import os

# Professional Canva-style themes
PROFESSIONAL_THEMES = {
    "modern_blue": {
        "name": "Modern Blue",
        "primary": RGBColor(41, 128, 185),
        "secondary": RGBColor(52, 152, 219),
        "accent": RGBColor(241, 196, 15),
        "dark": RGBColor(44, 62, 80),
        "light": RGBColor(236, 240, 241),
        "background": RGBColor(255, 255, 255),
    },
    "sunset_gradient": {
        "name": "Sunset Gradient",
        "primary": RGBColor(255, 107, 107),
        "secondary": RGBColor(255, 184, 77),
        "accent": RGBColor(72, 52, 212),
        "dark": RGBColor(46, 64, 83),
        "light": RGBColor(253, 203, 110),
        "background": RGBColor(255, 250, 240),
    },
    "corporate_green": {
        "name": "Corporate Green",
        "primary": RGBColor(39, 174, 96),
        "secondary": RGBColor(46, 204, 113),
        "accent": RGBColor(230, 126, 34),
        "dark": RGBColor(44, 62, 80),
        "light": RGBColor(236, 240, 241),
        "background": RGBColor(255, 255, 255),
    },
}

# Helper functions for transitions and animations
def add_slide_transition(slide, transition_type="fade", duration=500):
    """Add slide transition effect"""
    try:
        sld = slide._element
        p_ns = '{http://schemas.openxmlformats.org/presentationml/2006/main}'
        transition = etree.SubElement(sld, f'{p_ns}transition')
        transition.set('spd', 'med')
        transition.set('advTm', str(duration))
        
        if transition_type == "fade":
            fade = etree.SubElement(transition, f'{p_ns}fade')
            fade.set('thruBlk', '0')
        elif transition_type == "push":
            push = etree.SubElement(transition, f'{p_ns}push')
            push.set('dir', 'l')
        elif transition_type == "wipe":
            wipe = etree.SubElement(transition, f'{p_ns}wipe')
            wipe.set('dir', 'l')
        return True
    except Exception as e:
        print(f"⚠️ Transition error: {e}")
        return False

def add_shape_animations(slide, animation_type="fade"):
    """Add entrance animations to shapes (simplified version for Kaggle)"""
    # Simplified implementation for Kaggle notebook
    return True  # Animations added via XML in full version

def generate_slides_with_ai(topic, num_slides, text_model, text_tokenizer):
    """Generate high-quality slide content using AI model"""
    prompt = f"""Create a professional {num_slides}-slide PowerPoint presentation about "{topic}".

For each slide, provide:
1. A clear, concise title (5-10 words)
2. Content with 3-5 bullet points that are informative and specific to the topic

Format your response EXACTLY like this example:

SLIDE 1
Title: Introduction to {topic}
Content:
• First key point about the topic
• Second important detail
• Third crucial aspect
• Why this topic matters

SLIDE 2
Title: [Next topic aspect]
Content:
• Specific detail 1
• Specific detail 2
• Specific detail 3

Continue for all {num_slides} slides. Make the content informative, professional, and topic-specific. Avoid generic placeholders."""

    try:
        # Tokenize
        inputs = text_tokenizer(prompt, return_tensors="pt").to(text_model.device)
        
        # Generate with optimized settings
        with torch.no_grad():
            outputs = text_model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=1500,
                temperature=0.7,
                do_sample=True,
                pad_token_id=text_tokenizer.eos_token_id,
                eos_token_id=text_tokenizer.eos_token_id
            )
        
        # Decode response
        response = text_tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        
        # Parse the response into slide data
        slides = []
        current_slide = None
        
        for line in response.split('\n'):
            line = line.strip()
            
            if line.startswith('SLIDE '):
                if current_slide and current_slide.get('title') and current_slide.get('content'):
                    slides.append(current_slide)
                current_slide = {'title': '', 'content': ''}
            elif line.startswith('Title:'):
                if current_slide is not None:
                    current_slide['title'] = line.replace('Title:', '').strip()
            elif line.startswith('Content:'):
                if current_slide is not None:
                    current_slide['content'] = ''
            elif line.startswith('•') and current_slide is not None:
                if current_slide['content']:
                    current_slide['content'] += '\n'
                current_slide['content'] += line
        
        # Add the last slide
        if current_slide and current_slide.get('title') and current_slide.get('content'):
            slides.append(current_slide)
        
        # Ensure we have the right number of slides
        if len(slides) < num_slides:
            print(f"⚠️ AI generated {len(slides)} slides, padding to {num_slides}")
            # Use fallback for missing slides
            slides.extend(generate_slides_from_topic_fallback(topic, num_slides - len(slides)))
        elif len(slides) > num_slides:
            slides = slides[:num_slides]
        
        print(f"✅ AI generated {len(slides)} slides for {topic}")
        return slides
        
    except Exception as e:
        print(f"⚠️ AI generation failed: {e}, using fallback templates")
        return generate_slides_from_topic_fallback(topic, num_slides)

def generate_slides_from_topic_fallback(topic, num_slides=4):
    """Fallback template-based content generation"""
    slide_templates = {
        1: {
            "title": "Introduction",
            "content": f"What is {topic}?\n\n• Key characteristics\n• Main features\n• Why it matters\n• Real-world applications"
        },
        2: {
            "title": "Core Concepts",
            "content": "Fundamental Principles:\n\n• Core concept 1\n• Core concept 2\n• Core concept 3\n• Practical applications"
        },
        3: {
            "title": "Advanced Topics",
            "content": "Advanced Features:\n\n• Advanced topic 1\n• Advanced topic 2\n• Advanced topic 3\n• Best practices"
        },
        4: {
            "title": "Conclusion",
            "content": "Key Takeaways:\n\n• Summary point 1\n• Summary point 2\n• Summary point 3\n• Next steps"
        },
    }
    
    slides = []
    for i in range(1, min(num_slides, 5)):
        if i in slide_templates:
            slides.append(slide_templates[i])
    return slides

# Wrapper function for backward compatibility
def generate_slides_from_topic(topic, num_slides=4):
    """Generate slide content - uses AI if available, otherwise templates"""
    # This will be called with text_model passed explicitly
    return generate_slides_from_topic_fallback(topic, num_slides)

def create_powerpoint(title, slides_data, theme="modern_blue", output_path="/kaggle/working/presentation.pptx", add_animations=True, add_footer=True):
    """Create professional PowerPoint with Canva-style themes"""
    prs = Presentation()
    prs.slide_width = Inches(10)
    prs.slide_height = Inches(7.5)
    
    # Get theme
    theme_key = theme.lower().replace(" ", "_")
    if theme_key not in PROFESSIONAL_THEMES:
        theme_key = "modern_blue"
    colors = PROFESSIONAL_THEMES[theme_key]
    
    # Title slide
    slide = prs.slides.add_slide(prs.slide_layouts[6])
    background = slide.background
    fill = background.fill
    fill.solid()
    fill.fore_color.rgb = colors["background"]
    
    # Decorative shapes
    circle = slide.shapes.add_shape(MSO_SHAPE.OVAL, Inches(6.5), Inches(-1), Inches(4.5), Inches(4.5))
    circle.fill.solid()
    circle.fill.fore_color.rgb = colors["accent"]
    circle.line.fill.background()
    circle.fill.transparency = 0.3
    
    # Title
    title_box = slide.shapes.add_textbox(Inches(0.5), Inches(2.5), Inches(9), Inches(1.5))
    title_frame = title_box.text_frame
    title_frame.text = title
    title_para = title_frame.paragraphs[0]
    title_para.font.size = Pt(54)
    title_para.font.bold = True
    title_para.font.color.rgb = colors["primary"]
    title_para.alignment = PP_ALIGN.CENTER
    
    # Subtitle
    subtitle_box = slide.shapes.add_textbox(Inches(1), Inches(4.2), Inches(8), Inches(0.8))
    subtitle_frame = subtitle_box.text_frame
    current_date = datetime.now().strftime("%B %Y")
    subtitle_frame.text = f"Professional Presentation | {current_date}"
    subtitle_para = subtitle_frame.paragraphs[0]
    subtitle_para.font.size = Pt(24)
    subtitle_para.font.color.rgb = colors["dark"]
    subtitle_para.alignment = PP_ALIGN.CENTER
    
    if add_animations:
        add_slide_transition(slide, "fade", 800)
    
    # Content slides
    for idx, slide_info in enumerate(slides_data):
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        
        # Background
        background = slide.background
        fill = background.fill
        fill.solid()
        fill.fore_color.rgb = colors["background"]
        
        # Header bar
        header_bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(0), Inches(10), Inches(1.2))
        header_bar.fill.solid()
        header_bar.fill.fore_color.rgb = colors["primary"]
        header_bar.line.fill.background()
        
        # Title
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(7), Inches(0.8))
        title_frame = title_box.text_frame
        title_frame.text = slide_info['title']
        title_para = title_frame.paragraphs[0]
        title_para.font.size = Pt(36)
        title_para.font.bold = True
        title_para.font.color.rgb = RGBColor(255, 255, 255)
        
        # Image placeholder
        image_placeholder = slide.shapes.add_shape(MSO_SHAPE.ROUNDED_RECTANGLE, Inches(7.5), Inches(1.8), Inches(2.2), Inches(2.2))
        image_placeholder.fill.solid()
        image_placeholder.fill.fore_color.rgb = colors["light"]
        image_placeholder.line.color.rgb = colors["secondary"]
        image_placeholder.line.width = Pt(2)
        
        # Content
        content_box = slide.shapes.add_textbox(Inches(0.7), Inches(2), Inches(6.5), Inches(4.5))
        text_frame = content_box.text_frame
        text_frame.word_wrap = True
        
        content_lines = slide_info['content'].split('\n')
        for i, line in enumerate(content_lines):
            if not line.strip():
                continue
            
            if i == 0:
                p = text_frame.paragraphs[0]
            else:
                p = text_frame.add_paragraph()
            
            clean_line = line.strip().lstrip('•').strip()
            if line.strip().startswith('•') or '•' in line:
                p.text = f"◆  {clean_line}"
                p.font.color.rgb = colors["dark"]
                p.font.size = Pt(20)
                p.space_before = Pt(8)
            else:
                p.text = clean_line
                p.font.color.rgb = colors["secondary"]
                p.font.size = Pt(22)
                p.font.bold = True
                p.space_before = Pt(12)
        
        if add_footer:
            footer_bar = slide.shapes.add_shape(MSO_SHAPE.RECTANGLE, Inches(0), Inches(7.1), Inches(10), Inches(0.4))
            footer_bar.fill.solid()
            footer_bar.fill.fore_color.rgb = colors["light"]
            footer_bar.line.fill.background()
            
            footer_right = slide.shapes.add_textbox(Inches(8.5), Inches(7.15), Inches(1.2), Inches(0.3))
            footer_right_frame = footer_right.text_frame
            footer_right_frame.text = f"Slide {idx + 1}/{len(slides_data)}"
            footer_right_para = footer_right_frame.paragraphs[0]
            footer_right_para.font.size = Pt(10)
            footer_right_para.font.color.rgb = colors["dark"]
            footer_right_para.alignment = PP_ALIGN.RIGHT
        
        if add_animations:
            if idx % 2 == 0:
                add_slide_transition(slide, "push", 600)
            else:
                add_slide_transition(slide, "wipe", 600)
    
    # Save
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    prs.save(output_path)
    return output_path

print("✅ PowerPoint generator functions loaded!")
print("🤖 AI-powered content generation enabled using Qwen2.5-7B!")
print("📊 Available themes: modern_blue, sunset_gradient, corporate_green")

## 1.5️⃣ Configure GPU Memory Management

Set PyTorch CUDA allocator to reduce fragmentation

In [ ]:
import os
import gc

# Configure PyTorch CUDA allocator to reduce memory fragmentation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Clear any existing GPU cache
gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"✅ GPU memory configuration set")
        print(f"   Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
except:
    pass

In [ ]:
from pyngrok import ngrok

# Uses NGROK_AUTH_TOKEN set in the config cell above
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print(f"✅ Ngrok authenticated")
else:
    print("ℹ️  No NGROK_AUTH_TOKEN — ngrok will run without auth (limited tunnels)")
    print("   Get free token at https://ngrok.com")


## 3️⃣ Select Model Configuration

With Kaggle's 30GB RAM, you can use larger models!

In [ ]:
import os

# ============================================================
# CHOOSE YOUR LLM PROVIDER
# ============================================================
#
#  ┌──────────────┬────────────────────────────────────────────┐
#  │ Provider     │ What to set                                │
#  ├──────────────┼────────────────────────────────────────────┤
#  │ transformers │ Nothing — uses the GPU model loaded below  │  ← Kaggle default
#  │ ollama       │ OLLAMA_BASE_URL pointing to your server    │
#  │ groq         │ GROQ_API_KEY from console.groq.com (free)  │
#  │ huggingface  │ HF_API_KEY from huggingface.co (free)     │
#  │ openai       │ OPENAI_API_KEY (paid)                      │
#  └──────────────┴────────────────────────────────────────────┘
#
# On Kaggle with GPU: keep LLM_PROVIDER = "transformers"
# No GPU / just want fast free cloud: use "groq" + free key

LLM_PROVIDER = "transformers"   # ← change this to switch providers

# ── Groq (free, fastest cloud) ─────────────────────────────
GROQ_API_KEY = ""               # get free key → https://console.groq.com
GROQ_MODEL   = "llama-3.3-70b-versatile"   # also: mixtral-8x7b-32768, gemma2-9b-it

# ── HuggingFace Inference API (free tier) ──────────────────
HF_API_KEY   = ""               # get free token → https://huggingface.co/settings/tokens
HF_MODEL     = "mistralai/Mistral-7B-Instruct-v0.3"

# ── Ollama (local server) ──────────────────────────────────
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL    = "llama3"      # or: mistral, qwen2.5, phi3, deepseek-r1

# ── Local transformers model (loaded by cells 4+5 below) ───
# >>> CogAgent-9B: GUI agent model that sees screenshots and generates actions <<<
VISION_MODEL = "THUDM/cogagent-9b-20241220"   # CogAgent for GUI agent tasks (~5GB 4bit)
TEXT_MODEL   = "Qwen/Qwen2.5-7B-Instruct"     # main reasoning model

# Alternative vision models (uncomment to switch):
# VISION_MODEL = "Qwen/Qwen2-VL-2B-Instruct"    # lighter vision model (fits 15GB GPU)
# VISION_MODEL = "Qwen/Qwen2-VL-7B-Instruct"  # larger (needs 30GB+)
# TEXT_MODEL   = "Qwen/Qwen2.5-14B-Instruct"

USE_4BIT = True    # 4-bit quantization saves VRAM

# ── Ngrok (tunnel for local → internet) ─────────────────────
NGROK_AUTH_TOKEN = ""   # free token → https://ngrok.com

# ── Write all settings to environment so langchain_backend.py picks them up ──
os.environ["LLM_PROVIDER"]    = LLM_PROVIDER
os.environ["GROQ_API_KEY"]    = GROQ_API_KEY
os.environ["GROQ_MODEL"]      = GROQ_MODEL
os.environ["HF_API_KEY"]      = HF_API_KEY
os.environ["HF_MODEL"]        = HF_MODEL
os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
os.environ["OLLAMA_MODEL"]    = OLLAMA_MODEL
os.environ["HF_LOCAL_MODEL"]  = TEXT_MODEL      # transformers provider uses this

print(f"✅ Configuration set")
print(f"   LLM Provider : {LLM_PROVIDER}")
print(f"   Vision Model : {VISION_MODEL}")
print(f"   Text Model   : {TEXT_MODEL}")
print(f"   4-bit Quant  : {USE_4BIT}")
print()
if LLM_PROVIDER == "groq" and not GROQ_API_KEY:
    print("⚠️  GROQ_API_KEY is empty — get free key at https://console.groq.com")
if LLM_PROVIDER == "huggingface" and not HF_API_KEY:
    print("⚠️  HF_API_KEY is empty — get free token at https://huggingface.co/settings/tokens")
if LLM_PROVIDER == "transformers":
    print("ℹ️  Using local GPU model — run cells 4 & 5 to load it before starting the server")

## 3.5️⃣ Check Available RAM

In [ ]:
import psutil

ram_gb = psutil.virtual_memory().total / (1024**3)
ram_available = psutil.virtual_memory().available / (1024**3)

print(f"💾 Total RAM: {ram_gb:.1f} GB")
print(f"💾 Available RAM: {ram_available:.1f} GB")

if ram_gb > 25:
    print("\n✅ You have Kaggle's 30GB RAM! Perfect for GLM-4!")
else:
    print("\n⚠️ RAM seems low. Make sure GPU is enabled in Kaggle settings.")

## 4️⃣ Load Vision Model

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoProcessor, BitsAndBytesConfig
from transformers.generation.logits_process import LogitsProcessor
from PIL import Image
import base64
from io import BytesIO

# ─── InvalidScoreLogitsProcessor (from CogAgent's modeling_chatglm.py) ───
# This is a CRITICAL safety net: CogAgent can produce nan/inf logits,
# especially with quantized weights. This processor catches them.
class InvalidScoreLogitsProcessor(LogitsProcessor):
    """Catches nan/inf in logits and replaces with safe values."""
    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if torch.isnan(scores).any() or torch.isinf(scores).any():
            scores.zero_()
            scores[..., 198] = 5e4
        return scores

print(f"🔄 Loading Vision Model: {VISION_MODEL}...")
print(f"📊 Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

# Clear GPU cache before loading vision model
gc.collect()
torch.cuda.empty_cache()

# Quantization config
bnb_config = None
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

# ═══════════════════════════════════════════════════════════
#  CogAgent-9B-20241220 — GUI Agent (screenshot → actions)
# ═══════════════════════════════════════════════════════════
if "cogagent" in VISION_MODEL.lower():
    print("📦 Loading CogAgent-9B (GUI agent model)...")
    
    # Step 1: Load tokenizer
    vision_tokenizer = AutoTokenizer.from_pretrained(
        VISION_MODEL,
        trust_remote_code=True,
        padding_side="left"
    )
    print(f"   ✅ Tokenizer loaded (vocab: {len(vision_tokenizer)})")
    
    # Step 2: Load model
    vision_model = AutoModelForCausalLM.from_pretrained(
        VISION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    ).eval()
    print(f"   ✅ Model loaded: {type(vision_model).__name__}")
    
    # Step 3: Load image processor from AutoProcessor
    # The model's EVA2CLIPModel needs images preprocessed correctly.
    # AutoProcessor handles: resize, normalize, convert to tensor.
    try:
        _full_processor = AutoProcessor.from_pretrained(VISION_MODEL, trust_remote_code=True)
        # The processor has an image_processor component for image preprocessing
        if hasattr(_full_processor, 'image_processor') and _full_processor.image_processor is not None:
            cogagent_image_processor = _full_processor.image_processor
            print(f"   ✅ Image processor loaded from AutoProcessor")
        else:
            cogagent_image_processor = _full_processor
            print(f"   ✅ Using full processor for image preprocessing")
    except Exception as e:
        print(f"   ⚠️ AutoProcessor failed: {e}")
        print(f"   ℹ️ Will use manual image preprocessing")
        cogagent_image_processor = None
    
    # Step 4: Read special token IDs from config
    boi_token_id = getattr(vision_model.config, 'boi_token_id', None)
    eoi_token_id = getattr(vision_model.config, 'eoi_token_id', None)
    vision_config = getattr(vision_model.config, 'vision_config', {})
    image_size = vision_config.get("image_size", 1120)
    patch_size = vision_config.get("patch_size", 14)
    num_patches = (image_size // patch_size // 2) ** 2
    
    print(f"   ℹ️ BOI token ID: {boi_token_id}")
    print(f"   ℹ️ EOI token ID: {eoi_token_id}")
    print(f"   ℹ️ Image size: {image_size}, patches: {num_patches}")
    
    # Mark vision type for inference
    vision_processor = None  # signals CogAgent path (not Qwen2-VL)
    VISION_TYPE = "cogagent"
    
    # Step 5: Warm-up inference (text-only, no image — safe)
    print("🔥 Running warm-up inference (text-only)...")
    try:
        warmup_text = "<|user|>\nHello<|assistant|>\n"
        warmup_ids = vision_tokenizer.encode(warmup_text, return_tensors="pt")
        if warmup_ids is not None:
            warmup_ids = warmup_ids.to(vision_model.device)
            with torch.no_grad():
                warmup_out = vision_model.generate(
                    input_ids=warmup_ids,
                    max_new_tokens=5,
                    do_sample=False,
                    logits_processor=[InvalidScoreLogitsProcessor()],
                    pad_token_id=vision_tokenizer.eos_token_id or 128002,
                )
            response = vision_tokenizer.decode(warmup_out[0][warmup_ids.shape[-1]:], skip_special_tokens=True)
            print(f"   ✅ Warm-up OK: '{response[:50]}...'")
        else:
            print(f"   ⚠️ Tokenizer returned None — skipping warm-up")
    except Exception as e:
        print(f"   ⚠️ Warm-up failed: {e}")
        print(f"   ℹ️ This is OK — model is loaded, warm-up is optional")
    
    print(f"\n✅ CogAgent-9B Ready!")

# ═══════════════════════════════════════════════════════════
#  GLM-4V
# ═══════════════════════════════════════════════════════════
elif "glm-4v" in VISION_MODEL.lower():
    vision_tokenizer = AutoTokenizer.from_pretrained(
        VISION_MODEL,
        trust_remote_code=True
    )
    vision_model = AutoModelForCausalLM.from_pretrained(
        VISION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    ).eval()
    vision_processor = None
    cogagent_image_processor = None
    VISION_TYPE = "glm4v"
    print("✅ GLM-4V Loaded!")

# ═══════════════════════════════════════════════════════════
#  Qwen2-VL
# ═══════════════════════════════════════════════════════════
elif "qwen" in VISION_MODEL.lower() and "vl" in VISION_MODEL.lower():
    from transformers import Qwen2VLForConditionalGeneration
    
    vision_model = Qwen2VLForConditionalGeneration.from_pretrained(
        VISION_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    vision_processor = AutoProcessor.from_pretrained(VISION_MODEL, trust_remote_code=True)
    vision_tokenizer = None
    cogagent_image_processor = None
    VISION_TYPE = "qwen"
    print("✅ Qwen2-VL Loaded!")

else:
    print(f"⚠️ Unknown vision model: {VISION_MODEL}")
    vision_model = None
    vision_tokenizer = None
    vision_processor = None
    cogagent_image_processor = None
    VISION_TYPE = "none"

print(f"📊 RAM after vision model: {psutil.virtual_memory().available / (1024**3):.1f} GB available")

## 5️⃣ Load Text Model

In [ ]:
print(f"🔄 Loading Text Model: {TEXT_MODEL}...")
print(f"📊 Available RAM: {psutil.virtual_memory().available / (1024**3):.1f} GB")

# Clear GPU cache before loading text model
gc.collect()
torch.cuda.empty_cache()

text_tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL,
    trust_remote_code=True
)

text_model = AutoModelForCausalLM.from_pretrained(
    TEXT_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,
    trust_remote_code=True,
    low_cpu_mem_usage=True
).eval()

print("✅ Text Model Loaded!")
print(f"📊 RAM after both models: {psutil.virtual_memory().available / (1024**3):.1f} GB available")

ram_left = psutil.virtual_memory().available / (1024**3)
if ram_left < 2:
    print("\n⚠️ WARNING: Low RAM! Session may crash soon.")
else:
    print(f"\n✅ {ram_left:.1f}GB RAM remaining - looking good!")

## 6️⃣ Define Available Tools

In [ ]:
# Define available actions/tools for JARVIS
AVAILABLE_TOOLS = [
    {"type": "function", "function": {"name": "open_application", "description": "Opens an application by name (e.g., 'notepad', 'word', 'chrome', 'calculator')", "parameters": {"type": "object", "properties": {"app_name": {"type": "string", "description": "Name of the application to open"}}, "required": ["app_name"]}}},
    {"type": "function", "function": {"name": "type_text", "description": "Types text into the currently focused application", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "The text to type"}}, "required": ["text"]}}},
    {"type": "function", "function": {"name": "press_key", "description": "Presses a keyboard key or key combination (e.g., 'enter', 'ctrl+s', 'escape', 'tab')", "parameters": {"type": "object", "properties": {"key": {"type": "string", "description": "Key or key combination to press"}}, "required": ["key"]}}},
    {"type": "function", "function": {"name": "wait", "description": "Wait/delay for a specified duration in milliseconds", "parameters": {"type": "object", "properties": {"ms": {"type": "integer", "description": "Duration to wait in milliseconds"}}, "required": ["ms"]}}},
    {"type": "function", "function": {"name": "click_at", "description": "Click at specific screen coordinates", "parameters": {"type": "object", "properties": {"x": {"type": "integer", "description": "X coordinate"}, "y": {"type": "integer", "description": "Y coordinate"}}, "required": ["x", "y"]}}},
    {"type": "function", "function": {"name": "move_mouse", "description": "Move mouse to coordinates", "parameters": {"type": "object", "properties": {"x": {"type": "integer"}, "y": {"type": "integer"}}, "required": ["x", "y"]}}},
    {"type": "function", "function": {"name": "scroll", "description": "Scroll the page", "parameters": {"type": "object", "properties": {"direction": {"type": "string", "enum": ["up", "down"]}, "amount": {"type": "integer"}}, "required": ["direction"]}}},
    {"type": "function", "function": {"name": "create_file", "description": "Create a new file with optional content", "parameters": {"type": "object", "properties": {"file_path": {"type": "string", "description": "Full path to create the file"}, "content": {"type": "string", "description": "Content to write to the file (optional)"}}, "required": ["file_path"]}}},
    {"type": "function", "function": {"name": "create_folder", "description": "Create a new folder/directory", "parameters": {"type": "object", "properties": {"folder_path": {"type": "string", "description": "Full path for the new folder"}}, "required": ["folder_path"]}}},
    {"type": "function", "function": {"name": "delete_file", "description": "Delete a file or folder", "parameters": {"type": "object", "properties": {"file_path": {"type": "string", "description": "Path to file or folder to delete"}}, "required": ["file_path"]}}},
    {"type": "function", "function": {"name": "open_url", "description": "Open a URL in default browser", "parameters": {"type": "object", "properties": {"url": {"type": "string", "description": "URL to open"}}, "required": ["url"]}}},
    {"type": "function", "function": {"name": "search_web", "description": "Search Google for a query", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "Search query"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "run_command", "description": "Run a system command (e.g., for Bluetooth control, system settings)", "parameters": {"type": "object", "properties": {"command": {"type": "string", "description": "Command to execute"}}, "required": ["command"]}}},
    {"type": "function", "function": {"name": "ppt_apply_theme", "description": "Apply a design theme to PowerPoint presentation (must have PowerPoint open with active presentation)", "parameters": {"type": "object", "properties": {"theme_name": {"type": "string", "description": "Theme name: 'ion', 'facet', 'integral', 'ion_boardroom', 'office_theme', 'slice', 'wisp', 'organic', 'retrospect', 'dividend'"}}, "required": ["theme_name"]}}},
    {"type": "function", "function": {"name": "ppt_add_animation", "description": "Add entrance animation to current slide in PowerPoint", "parameters": {"type": "object", "properties": {"animation_type": {"type": "string", "description": "Animation: 'fade', 'fly_in', 'wipe', 'split', 'appear', 'zoom', 'swivel', 'bounce'"}, "apply_to_all": {"type": "boolean", "description": "Apply to all slides (default: false)"}}, "required": ["animation_type"]}}},
    {"type": "function", "function": {"name": "ppt_change_layout", "description": "Change layout of current slide in PowerPoint", "parameters": {"type": "object", "properties": {"layout_name": {"type": "string", "description": "Layout: 'title_slide', 'title_and_content', 'section_header', 'two_content', 'comparison', 'title_only', 'blank', 'content_with_caption', 'picture_with_caption'"}}, "required": ["layout_name"]}}},
    
    # Word automation tools
    {"type": "function", "function": {"name": "word_create_document", "description": "Create a new Word document with title and initial content", "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "Document title"}, "content": {"type": "string", "description": "Initial content/body text"}}, "required": ["title"]}}},
    {"type": "function", "function": {"name": "word_add_paragraph", "description": "Add a paragraph to the current Word document with full styling", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "Paragraph text"}, "font_size": {"type": "integer", "description": "Font size (default: 12)"}, "font_name": {"type": "string", "description": "Font family: Arial, Calibri, Times New Roman, Georgia, Verdana, etc."}, "bold": {"type": "boolean", "description": "Make text bold"}, "italic": {"type": "boolean", "description": "Make text italic"}, "underline": {"type": "boolean", "description": "Underline text"}, "color": {"type": "string", "description": "Text color: red, blue, green, black, orange, purple, yellow, etc."}, "alignment": {"type": "string", "description": "Text alignment: left, center, right, justify"}}, "required": ["text"]}}},
    {"type": "function", "function": {"name": "word_add_heading", "description": "Add a heading to the current Word document", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "Heading text"}, "level": {"type": "integer", "description": "Heading level 1-3 (default: 1)"}, "font_name": {"type": "string", "description": "Font family (optional)"}, "font_size": {"type": "integer", "description": "Font size (optional)"}, "color": {"type": "string", "description": "Text color (optional)"}}, "required": ["text"]}}},
    {"type": "function", "function": {"name": "word_insert_table", "description": "Insert a table into the current Word document", "parameters": {"type": "object", "properties": {"rows": {"type": "integer", "description": "Number of rows"}, "columns": {"type": "integer", "description": "Number of columns"}}, "required": ["rows", "columns"]}}},
    {"type": "function", "function": {"name": "word_apply_theme", "description": "Apply a color theme to the Word document", "parameters": {"type": "object", "properties": {"theme": {"type": "string", "description": "Theme name: 'blue', 'green', 'orange', 'red', 'purple'"}}, "required": ["theme"]}}},
    {"type": "function", "function": {"name": "word_save", "description": "Save the current Word document to a file", "parameters": {"type": "object", "properties": {"filename": {"type": "string", "description": "Full path or filename to save the document"}}, "required": ["filename"]}}},
    {"type": "function", "function": {"name": "word_remove_table_borders", "description": "Remove borders from tables in the Word document - either a specific table or all tables", "parameters": {"type": "object", "properties": {"table_number": {"type": "integer", "description": "Table number to remove borders from (optional - if omitted, removes from all tables)"}}, "required": []}}},
    {"type": "function", "function": {"name": "word_change_font", "description": "Change the font in the Word document - either for entire document or specific paragraph range", "parameters": {"type": "object", "properties": {"font_name": {"type": "string", "description": "Font family (e.g., Arial, Calibri)"}, "font_size": {"type": "integer", "description": "Font size (optional)"}, "start_paragraph": {"type": "integer", "description": "Starting paragraph number (optional - if omitted, applies to entire document)"}, "end_paragraph": {"type": "integer", "description": "Ending paragraph number (optional)"}}, "required": ["font_name"]}}},
    {"type": "function", "function": {"name": "word_clear_formatting", "description": "Remove all formatting (bold, italic, underline, color) - either from entire document or specific paragraph range", "parameters": {"type": "object", "properties": {"start_paragraph": {"type": "integer", "description": "Starting paragraph number (optional - if omitted, clears entire document)"}, "end_paragraph": {"type": "integer", "description": "Ending paragraph number (optional)"}}, "required": []}}},
    {"type": "function", "function": {"name": "word_change_color", "description": "Change text color in the Word document - either for entire document or specific paragraph range", "parameters": {"type": "object", "properties": {"color": {"type": "string", "description": "Color name: red, blue, green, black, white, yellow, orange, purple, etc."}, "start_paragraph": {"type": "integer", "description": "Starting paragraph number (optional - if omitted, applies to entire document)"}, "end_paragraph": {"type": "integer", "description": "Ending paragraph number (optional)"}}, "required": ["color"]}}},
    
    # Excel automation tools
    {"type": "function", "function": {"name": "excel_create_workbook", "description": "Create a new Excel workbook", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "excel_write_cell", "description": "Write data to a specific cell in Excel", "parameters": {"type": "object", "properties": {"row": {"type": "integer", "description": "Row number (1-based)"}, "col": {"type": "integer", "description": "Column number (1-based)"}, "value": {"type": "string", "description": "Value to write"}}, "required": ["row", "col", "value"]}}},
    {"type": "function", "function": {"name": "excel_write_data", "description": "Write array of data to Excel (starting from A1)", "parameters": {"type": "object", "properties": {"data": {"type": "array", "description": "2D array of data [[row1], [row2], ...]"}}, "required": ["data"]}}},
    {"type": "function", "function": {"name": "excel_add_worksheet", "description": "Add a new worksheet to the Excel workbook", "parameters": {"type": "object", "properties": {"name": {"type": "string", "description": "Worksheet name (optional)"}}, "required": []}}},
    {"type": "function", "function": {"name": "excel_create_chart", "description": "Create a chart from selected data", "parameters": {"type": "object", "properties": {"chart_type": {"type": "string", "description": "Chart type: 'column', 'bar', 'line', 'pie', 'area'"}}, "required": ["chart_type"]}}},
    {"type": "function", "function": {"name": "excel_remove_borders", "description": "Remove borders from cells - either from a specific range or from the entire sheet", "parameters": {"type": "object", "properties": {"start_row": {"type": "integer", "description": "Starting row (optional - if omitted, removes from entire sheet)"}, "start_col": {"type": "integer", "description": "Starting column (optional)"}, "end_row": {"type": "integer", "description": "Ending row (optional)"}, "end_col": {"type": "integer", "description": "Ending column (optional)"}}, "required": []}}},
    {"type": "function", "function": {"name": "excel_clear_formatting", "description": "Clear all formatting (fonts, colors, borders) from cells - either from a specific range or from the entire sheet", "parameters": {"type": "object", "properties": {"start_row": {"type": "integer", "description": "Starting row (optional - if omitted, clears entire sheet)"}, "start_col": {"type": "integer", "description": "Starting column (optional)"}, "end_row": {"type": "integer", "description": "Ending row (optional)"}, "end_col": {"type": "integer", "description": "Ending column (optional)"}}, "required": []}}},
    {"type": "function", "function": {"name": "excel_remove_background", "description": "Remove background colors from cells - either from a specific range or from the entire sheet", "parameters": {"type": "object", "properties": {"start_row": {"type": "integer", "description": "Starting row (optional - if omitted, removes from entire sheet)"}, "start_col": {"type": "integer", "description": "Starting column (optional)"}, "end_row": {"type": "integer", "description": "Ending row (optional)"}, "end_col": {"type": "integer", "description": "Ending column (optional)"}}, "required": []}}},
    {"type": "function", "function": {"name": "excel_change_font", "description": "Change the font in Excel - either for entire sheet or specific range", "parameters": {"type": "object", "properties": {"font_name": {"type": "string", "description": "Font family (e.g., Arial, Calibri)"}, "font_size": {"type": "integer", "description": "Font size (optional)"}, "start_row": {"type": "integer", "description": "Starting row (optional - if omitted, applies to entire sheet)"}, "start_col": {"type": "integer", "description": "Starting column (optional)"}, "end_row": {"type": "integer", "description": "Ending row (optional)"}, "end_col": {"type": "integer", "description": "Ending column (optional)"}}, "required": ["font_name"]}}},
    
    # Publisher automation tools
    {"type": "function", "function": {"name": "publisher_create", "description": "Create a new Publisher publication", "parameters": {"type": "object", "properties": {"template_type": {"type": "string", "description": "Template type (default: 'blank')"}}, "required": []}}},
    {"type": "function", "function": {"name": "publisher_add_textbox", "description": "Add a text box to the Publisher publication", "parameters": {"type": "object", "properties": {"text": {"type": "string", "description": "Text content"}, "left": {"type": "integer", "description": "Left position (default: 100)"}, "top": {"type": "integer", "description": "Top position (default: 100)"}, "width": {"type": "integer", "description": "Width (default: 300)"}, "height": {"type": "integer", "description": "Height (default: 100)"}}, "required": ["text"]}}},
    {"type": "function", "function": {"name": "publisher_add_page", "description": "Add a new page to the Publisher publication", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "publisher_save", "description": "Save the Publisher publication to a file", "parameters": {"type": "object", "properties": {"filename": {"type": "string", "description": "Full path or filename to save (.pub)"}}, "required": ["filename"]}}},
    
    # OneNote automation tools
    {"type": "function", "function": {"name": "onenote_open", "description": "Open Microsoft OneNote application", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "onenote_create_page", "description": "Create a new page in OneNote", "parameters": {"type": "object", "properties": {"title": {"type": "string", "description": "Page title"}}, "required": ["title"]}}},
    {"type": "function", "function": {"name": "onenote_add_content", "description": "Add content to the current OneNote page", "parameters": {"type": "object", "properties": {"content": {"type": "string", "description": "Content to add"}}, "required": ["content"]}}},
    
    # Paint automation tools (cursor-based)
    {"type": "function", "function": {"name": "paint_open", "description": "Open Microsoft Paint application", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "paint_draw_line", "description": "Draw a line in Paint (requires Paint to be open)", "parameters": {"type": "object", "properties": {"x1": {"type": "integer"}, "y1": {"type": "integer"}, "x2": {"type": "integer"}, "y2": {"type": "integer"}}, "required": ["x1", "y1", "x2", "y2"]}}},
    {"type": "function", "function": {"name": "paint_select_tool", "description": "Select a drawing tool in Paint", "parameters": {"type": "object", "properties": {"tool": {"type": "string", "description": "Tool name: 'pencil', 'brush', 'eraser', 'rectangle', 'circle'"}}, "required": ["tool"]}}},
    {"type": "function", "function": {"name": "paint_select_color", "description": "Select a color in Paint", "parameters": {"type": "object", "properties": {"color": {"type": "string", "description": "Color name or hex code"}}, "required": ["color"]}}},
    
    # Whiteboard automation tools (cursor-based)
    {"type": "function", "function": {"name": "whiteboard_open", "description": "Open Microsoft Whiteboard application", "parameters": {"type": "object", "properties": {}, "required": []}}},
    
    # File system utilities
    {"type": "function", "function": {"name": "check_file_exists", "description": "Check if a file exists at the specified path and suggest numbered alternative if it does", "parameters": {"type": "object", "properties": {"filepath": {"type": "string", "description": "Full path to check (e.g., C:\\\\Users\\\\%USERNAME%\\\\Documents\\\\Report.docx)"}}, "required": ["filepath"]}}},
    {"type": "function", "function": {"name": "generate_unique_filename", "description": "Generate a unique filename by adding numbers if file exists (Report.docx → Report_2.docx)", "parameters": {"type": "object", "properties": {"filepath": {"type": "string", "description": "Desired file path"}, "content_description": {"type": "string", "description": "Brief description of document content for smart naming if no name provided"}}, "required": ["filepath"]}}},
    {"type": "function", "function": {"name": "search_files", "description": "Search for files in Documents, Desktop, Downloads by name. Returns list of matching files with full paths.", "parameters": {"type": "object", "properties": {"search_term": {"type": "string", "description": "Filename or part of filename to search for"}, "file_type": {"type": "string", "description": "File type: 'word', 'excel', 'powerpoint', 'pdf', 'text', 'publisher', 'all' (optional)"}, "search_location": {"type": "string", "description": "Specific folder to search in (optional - defaults to Documents, Desktop, Downloads)"}}, "required": ["search_term"]}}},
    {"type": "function", "function": {"name": "word_open_document", "description": "Open an existing Word document for editing", "parameters": {"type": "object", "properties": {"filepath": {"type": "string", "description": "Full path to the Word document"}}, "required": ["filepath"]}}},
    {"type": "function", "function": {"name": "excel_open_workbook", "description": "Open an existing Excel workbook for editing", "parameters": {"type": "object", "properties": {"filepath": {"type": "string", "description": "Full path to the Excel workbook"}}, "required": ["filepath"]}}},
    
    # System Management Tools (17 tools for Windows system control)
    {"type": "function", "function": {"name": "clear_temp_files", "description": "Clear temporary files and cache to free up disk space. Clears Windows Temp folders, browser cache, Windows Update cache, thumbnail cache. Returns amount of space freed in MB. Safe operation - only removes temporary data.", "parameters": {"type": "object", "properties": {"include_cache": {"type": "boolean", "description": "Include browser cache and Windows cache (default: true). Set false to only clear temp files."}}, "required": []}}},
    {"type": "function", "function": {"name": "set_wallpaper", "description": "Change desktop wallpaper/background image. Image must be JPG, PNG, or BMP format. Uses Windows API for instant wallpaper change.", "parameters": {"type": "object", "properties": {"image_path": {"type": "string", "description": "Full path to wallpaper image file (e.g., 'C:\\\\Pictures\\\\wallpaper.jpg')"}}, "required": ["image_path"]}}},
    {"type": "function", "function": {"name": "toggle_wifi", "description": "Enable or disable WiFi adapter. Turns wireless network on/off. Requires admin rights on some systems.", "parameters": {"type": "object", "properties": {"enable": {"type": "boolean", "description": "true = turn WiFi ON, false = turn WiFi OFF"}}, "required": ["enable"]}}},
    {"type": "function", "function": {"name": "toggle_bluetooth", "description": "Enable or disable Bluetooth adapter. Turns Bluetooth service on/off. Requires admin rights.", "parameters": {"type": "object", "properties": {"enable": {"type": "boolean", "description": "true = turn Bluetooth ON, false = turn Bluetooth OFF"}}, "required": ["enable"]}}},
    {"type": "function", "function": {"name": "set_brightness", "description": "Adjust display brightness level. Works on laptops and monitors with DDC/CI support. Changes system brightness setting.", "parameters": {"type": "object", "properties": {"brightness": {"type": "integer", "description": "Brightness level from 0 (darkest) to 100 (brightest)"}}, "required": ["brightness"]}}},
    {"type": "function", "function": {"name": "get_battery_status", "description": "Get battery information for laptops: charge percentage, charging status, estimated time remaining. Returns detailed battery stats.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "set_resolution", "description": "Change screen resolution. Sets display width and height in pixels. Common resolutions: 1920x1080 (Full HD), 2560x1440 (2K), 3840x2160 (4K), 1366x768, 1280x720.", "parameters": {"type": "object", "properties": {"width": {"type": "integer", "description": "Screen width in pixels (e.g., 1920)"}, "height": {"type": "integer", "description": "Screen height in pixels (e.g., 1080)"}}, "required": ["width", "height"]}}},
    {"type": "function", "function": {"name": "get_system_info", "description": "Get comprehensive system information: OS version, CPU name, RAM capacity, system uptime, computer name. Returns all key system details.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "toggle_night_light", "description": "Enable or disable Windows Night Light mode (blue light filter). Night Light reduces blue light for better sleep. Works on Windows 10/11.", "parameters": {"type": "object", "properties": {"enable": {"type": "boolean", "description": "true = turn Night Light ON, false = turn Night Light OFF"}}, "required": ["enable"]}}},
    {"type": "function", "function": {"name": "empty_recycle_bin", "description": "Empty Windows Recycle Bin permanently. Deletes all files in Recycle Bin to free up disk space. WARNING: This action cannot be undone!", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "get_disk_space", "description": "Check disk space usage for all drives (C:, D:, E:, etc.). Returns total capacity, used space, free space, and usage percentage for each drive.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "set_volume", "description": "Adjust system volume level. Changes master audio volume for all applications. Uses Windows audio API.", "parameters": {"type": "object", "properties": {"volume": {"type": "integer", "description": "Volume level from 0 (mute) to 100 (maximum)"}}, "required": ["volume"]}}},
    {"type": "function", "function": {"name": "lock_computer", "description": "Lock the computer screen immediately. Same as Windows+L. User must enter password to unlock. Useful for quick security when stepping away.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "sleep_computer", "description": "Put computer into sleep/suspend mode. Saves power while keeping programs running. Quick resume to current state. WARNING: Make sure all work is saved first!", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "get_network_status", "description": "Get detailed network adapter information: active WiFi/Ethernet connections, adapter names, connection status, IP addresses, link speed.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "run_disk_cleanup", "description": "Launch Windows Disk Cleanup utility (cleanmgr.exe). Interactive tool to clean up system files, old Windows installations, temporary files, downloads. Opens GUI for user to select what to clean.", "parameters": {"type": "object", "properties": {}, "required": []}}},
    {"type": "function", "function": {"name": "toggle_windows_defender", "description": "Enable or disable Windows Defender real-time protection. WARNING: Requires admin rights. Disabling antivirus is RISKY - only do if user explicitly requests. May require confirmation.", "parameters": {"type": "object", "properties": {"enable": {"type": "boolean", "description": "true = turn Windows Defender ON, false = turn Windows Defender OFF"}}, "required": ["enable"]}}}
]

TOOLS_DESCRIPTION = "\n".join([f"- {t['function']['name']}: {t['function']['description']}" for t in AVAILABLE_TOOLS])
print(f"✅ Defined {len(AVAILABLE_TOOLS)} tools!")
print(f"   📁 File Management: 13 tools")
print(f"   🖥️  Application Management: 5 tools")
print(f"   ⚙️  System Management: 17 tools") 
print(f"   📝 Office Automation: 40+ tools")
print(f"   🎨 Paint/Whiteboard: 5 tools")
print("\nAvailable Actions:")
print(TOOLS_DESCRIPTION)


## 7️⃣ Vision & Action Functions

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
#  CogAgent Vision Inference — screenshot → action
# ═══════════════════════════════════════════════════════════════════════
#
# KEY INSIGHTS from analyzing CogAgent-9B-20241220 modeling_chatglm.py:
#
# 1. The model has NO build_conversation_input_ids() or chat() methods.
# 2. The model's forward() accepts: input_ids, images, attention_mask
# 3. input_ids MUST contain boi_token_id + placeholder + eoi_token_id
# 4. The model replaces that 3-token region with vision features internally
# 5. images is a tensor [batch, C, H, W] for EVA2CLIPModel
# 6. InvalidScoreLogitsProcessor catches nan/inf in logits (known issue)
# 7. prepare_inputs_for_generation() expects `images=` NOT `pixel_values=`
#
import re
import io

def cogagent_vision_act(screenshot_b64, goal, step_history=None, 
                        screen_width=1920, screen_height=1080):
    """
    Run CogAgent inference: screenshot + goal -> action.
    
    CogAgent-9B is a GUI agent model that sees screenshots and outputs
    grounded operations like CLICK(box=[[x1,y1,x2,y2]]), TYPE(text='...'), etc.
    """
    try:
        # -- 1. Format CogAgent prompt --
        platform_str = "(Platform: WIN)\n"
        format_str = "(Answer in Action-Operation-Sensitive format.)\n"
        
        history_str = "\nHistory steps: "
        if step_history:
            for idx, step in enumerate(step_history):
                if isinstance(step, dict):
                    step_text = step.get("step", step.get("description", ""))
                    action_text = step.get("action", "")
                    history_str += f"\n{idx}. {step_text}\t{action_text}"
                elif isinstance(step, str):
                    history_str += f"\n{idx}. {step}"
        
        query = f"Task: {goal}{history_str}\n{platform_str}{format_str}"
        print(f"  CogAgent query: {query[:120]}...")
        
        # -- 2. Build input_ids with image tokens --
        # CogAgent expects: ...<|begin_of_image|><placeholder><|end_of_image|>...
        # The assertion in forward(): eoi_token_pos - boi_token_pos == 2
        
        boi = getattr(vision_model.config, 'boi_token_id', None)
        eoi = getattr(vision_model.config, 'eoi_token_id', None)
        
        input_ids = None
        
        # Strategy A: Use tokenizer.apply_chat_template with image content
        if hasattr(vision_tokenizer, 'apply_chat_template'):
            try:
                # GLM-4V tokenizers support multimodal messages
                messages = [{
                    "role": "user",
                    "content": [
                        {"type": "image"},
                        {"type": "text", "text": query}
                    ]
                }]
                text = vision_tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                input_ids = vision_tokenizer.encode(text, return_tensors="pt")
                
                # Verify BOI/EOI tokens are present
                id_list = input_ids[0].tolist()
                if boi and boi in id_list and eoi and eoi in id_list:
                    boi_pos = id_list.index(boi)
                    eoi_pos = id_list.index(eoi)
                    if eoi_pos - boi_pos == 2:
                        print(f"   Strategy A OK: chat template (BOI@{boi_pos}, EOI@{eoi_pos})")
                    else:
                        print(f"   Strategy A: BOI-EOI gap is {eoi_pos - boi_pos}, expected 2. Trying B.")
                        input_ids = None
                else:
                    print(f"   Strategy A: BOI/EOI not found. Trying B.")
                    input_ids = None
            except Exception as e:
                print(f"   Strategy A failed: {e}")
                input_ids = None
        
        # Strategy B: Manual token construction
        if input_ids is None and boi is not None and eoi is not None:
            try:
                # Encode text-only to get template structure
                messages = [{"role": "user", "content": query}]
                text_only = vision_tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
                text_ids = vision_tokenizer.encode(text_only, add_special_tokens=False)
                
                # Placeholder token between BOI and EOI
                placeholder_id = vision_tokenizer.eos_token_id or 151329
                image_tokens = [boi, placeholder_id, eoi]
                
                # Find the first token of query content in encoded IDs
                query_start = vision_tokenizer.encode(query[:20], add_special_tokens=False)[:3]
                
                insert_pos = None
                for i in range(len(text_ids) - len(query_start) + 1):
                    if text_ids[i:i+len(query_start)] == query_start:
                        insert_pos = i
                        break
                
                if insert_pos is not None:
                    final_ids = text_ids[:insert_pos] + image_tokens + text_ids[insert_pos:]
                else:
                    # Fallback: insert after first few template tokens
                    final_ids = text_ids[:4] + image_tokens + text_ids[4:]
                
                input_ids = torch.tensor([final_ids])
                print(f"   Strategy B OK: manual insertion")
                        
            except Exception as e:
                print(f"   Strategy B failed: {e}")
        
        # Strategy C: Direct text encoding with special token names
        if input_ids is None:
            try:
                full_text = f"<|user|>\n<|begin_of_image|><|endoftext|><|end_of_image|>{query}<|assistant|>\n"
                input_ids = vision_tokenizer.encode(full_text, return_tensors="pt")
                print(f"   Strategy C: direct text encoding")
            except Exception as e:
                print(f"   All strategies failed: {e}")
                return {"action": "fail", "description": f"Failed to tokenize: {e}"}
        
        input_ids = input_ids.to(vision_model.device)
        print(f"   input_ids shape: {input_ids.shape}")
        
        # -- 3. Process screenshot image --
        image_data = base64.b64decode(screenshot_b64)
        pil_image = Image.open(io.BytesIO(image_data)).convert("RGB")
        
        images = None
        if cogagent_image_processor is not None:
            try:
                processed = cogagent_image_processor(pil_image, return_tensors="pt")
                if isinstance(processed, dict):
                    images = processed.get("pixel_values", processed.get("images"))
                else:
                    images = processed
                images = images.to(device=vision_model.device, dtype=torch.float16)
                print(f"   Image processed via processor: {images.shape}")
            except Exception as e:
                print(f"   Image processor failed: {e}, using manual")
                images = None
        
        if images is None:
            # Manual preprocessing for EVA2CLIP
            from torchvision import transforms
            img_size = getattr(vision_model.config, 'vision_config', {}).get("image_size", 1120)
            transform = transforms.Compose([
                transforms.Resize((img_size, img_size), 
                                  interpolation=transforms.InterpolationMode.BICUBIC),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.48145466, 0.4578275, 0.40821073],
                    std=[0.26862954, 0.26130258, 0.27577711]
                )
            ])
            images = transform(pil_image).unsqueeze(0).to(
                device=vision_model.device, dtype=torch.float16
            )
            print(f"   Manual image preprocessing: {images.shape}")
        
        # -- 4. Generate with safety net --
        attention_mask = torch.ones_like(input_ids)
        
        gen_kwargs = {
            "max_new_tokens": 512,
            "do_sample": False,
            "pad_token_id": 128002,
            "logits_processor": [InvalidScoreLogitsProcessor()],
        }
        
        print(f"   Running CogAgent inference...")
        with torch.no_grad():
            outputs = vision_model.generate(
                input_ids=input_ids,
                images=images,          # KEY: must be `images`, NOT `pixel_values`
                attention_mask=attention_mask,
                **gen_kwargs,
            )
        
        # Decode response
        new_tokens = outputs[0][input_ids.shape[-1]:]
        response = vision_tokenizer.decode(new_tokens, skip_special_tokens=True)
        print(f"   CogAgent response: {response[:200]}")
        
        torch.cuda.empty_cache()
        
        # -- 5. Parse CogAgent output -> action JSON --
        return _parse_cogagent_output(response, screen_width, screen_height)
        
    except Exception as e:
        print(f"CogAgent inference error: {e}")
        import traceback
        traceback.print_exc()
        torch.cuda.empty_cache()
        return {"action": "fail", "description": f"CogAgent error: {str(e)}"}


def _parse_cogagent_output(response, screen_width=1920, screen_height=1080):
    """Parse CogAgent output into structured action JSON.
    
    CogAgent outputs in format like:
    Status: continue
    Plan: Click the search box
    Action: Left click on the search box 
    Grounded Operation: CLICK(box=[[352,102,786,139]], element_info='Search')
    """
    grounded_match = re.search(r"Grounded Operation:\s*(.*?)(?:\n|$)", response)
    action_match = re.search(r"Action:\s*(.*?)(?:\n|$)", response)
    status_match = re.search(r"Status:\s*(.*?)(?:\n|$)", response)
    
    step = grounded_match.group(1).strip() if grounded_match else None
    action_desc = action_match.group(1).strip() if action_match else response.strip()
    status = status_match.group(1).strip().lower() if status_match else ""
    
    # Check if task is done
    if status in ("finish", "finished", "done", "completed"):
        return {"action": "done", "description": action_desc or "Task completed"}
    
    if not step:
        if any(w in response.lower() for w in ["done", "finish", "complet", "success"]):
            return {"action": "done", "description": response[:200]}
        return {"action": "fail", "description": f"No operation: {response[:200]}"}
    
    # Parse operation type
    op_match = re.match(r'(\w+)\s*\(', step)
    if not op_match:
        return {"action": "fail", "description": step}
    
    operation = op_match.group(1).upper()
    
    # Extract bounding box (CogAgent uses 0-1000 coordinate system)
    box_pattern = r"box=\[\[?(\d+),\s*(\d+),\s*(\d+),\s*(\d+)\]?\]"
    box_match = re.search(box_pattern, step)
    
    result = {"description": action_desc, "raw_step": step}
    
    def _box_center():
        """Convert 0-1000 box coords to screen pixel center."""
        x1, y1, x2, y2 = [int(v) for v in box_match.groups()]
        return (
            int((x1 + x2) / 2 * screen_width / 1000),
            int((y1 + y2) / 2 * screen_height / 1000)
        )
    
    if operation == "CLICK":
        result["action"] = "click"
        if box_match:
            result["x"], result["y"] = _box_center()
            
    elif operation == "TYPE":
        result["action"] = "type"
        text_match = re.search(r"text=['\"]([^'\"]*)['\"]", step)
        result["text"] = text_match.group(1) if text_match else ""
        if box_match:
            result["x"], result["y"] = _box_center()
            
    elif operation in ("SCROLL_DOWN", "SCROLL_UP"):
        result["action"] = "scroll"
        result["direction"] = "down" if "DOWN" in operation else "up"
        step_count = re.search(r"step_count=(\d+)", step)
        result["amount"] = int(step_count.group(1)) if step_count else 3
        if box_match:
            result["x"], result["y"] = _box_center()
            
    elif operation == "PRESS":
        result["action"] = "key_press"
        key_match = re.search(r"key=['\"]([^'\"]*)['\"]", step)
        result["key"] = key_match.group(1) if key_match else "enter"
        
    elif operation == "HOVER":
        result["action"] = "hover"
        if box_match:
            result["x"], result["y"] = _box_center()
            
    elif operation == "WAIT":
        result["action"] = "wait"
        
    elif operation in ("SELECT", "DRAG", "DOUBLE_CLICK", "RIGHT_CLICK"):
        result["action"] = operation.lower()
        if box_match:
            result["x"], result["y"] = _box_center()
    else:
        result["action"] = operation.lower()
        if box_match:
            result["x"], result["y"] = _box_center()
    
    print(f"   Parsed: {result.get('action')} at ({result.get('x','?')}, {result.get('y','?')})")
    return result


print("CogAgent inference functions loaded!")

In [ ]:
import json
import re

def analyze_screenshot(image_base64, query, screen_width, screen_height):
    """Analyze screenshot using loaded vision model - OPTIONAL, will work without vision"""
    try:
        # If no image provided, skip vision analysis
        if not image_base64:
            return "No screenshot provided"
        
        # Check if vision model is available
        if vision_model is None or vision_processor is None:
            return "Vision analysis unavailable (model not loaded)"
        
        # Prepare image
        image_data = base64.b64decode(image_base64)
        image = Image.open(io.BytesIO(image_data))
        
        # Vision query
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": query or "Describe what's on the screen in detail."}
                ]
            }
        ]
        
        # Prepare for inference
        try:
            text = vision_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)
            
            inputs = vision_processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt"
            ).to(vision_model.device)
            
            # Generate
            with torch.no_grad():
                generated_ids = vision_model.generate(**inputs, max_new_tokens=512)
                generated_ids_trimmed = [
                    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
                ]
                decoded = vision_processor.batch_decode(
                    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
                )[0]
            
            if decoded and decoded.strip():
                return decoded if decoded.strip() else "Desktop screen visible"
            else:
                return "Desktop screen (vision unavailable)"
        except Exception as e:
            print(f"Vision processing error: {e}")
            return "Desktop screen (vision unavailable)"
    except Exception as e:
        print(f"Screenshot analysis error: {e}")
        return "Desktop screen (vision unavailable)"

def generate_actions(user_message, screenshot_analysis, conversation_history):
    """Generate actions using Qwen2.5-7B with OPTIMIZED MEMORY USAGE for 15GB GPU"""
    try:
        # ===== MEMORY OPTIMIZATION: Clear GPU cache before generation =====
        import torch
        torch.cuda.empty_cache()
        gc.collect()
        
        # ===== ULTRA-COMPACT SYSTEM PROMPT (reduced from 11k to ~3k tokens) =====
        system_prompt = f"""You are JARVIS, a conversational desktop AI assistant. Generate helpful responses WITH actions.

AVAILABLE TOOLS ({len(AVAILABLE_TOOLS)} total):
{json.dumps(AVAILABLE_TOOLS, indent=0)}

CRITICAL RULES:
1. Output JSON: {{"message": "conversational response", "actions": [...], "requires_choice": false}}
2. Message should be CONVERSATIONAL and helpful, not just "executing" or "done"
3. Each action: {{"name": "tool_name", "parameters": {{...}}}}
4. For ALL file creation (PowerPoint/Word/Excel/Paint/Publisher/Whiteboard/OneNote/Notepad): Create first, then ASK where to save with requires_choice: true
5. If user says NO/CANCEL/NEVERMIND to saving: Acknowledge and discard the file (no save action)
6. If user says YES/SAVE without specifying location: Use Desktop and smart filename based on content/topic
7. If file already exists: Ask user to overwrite/rename/cancel with requires_choice: true
8. Use full Windows paths: C:\\Users\\%USERNAME%\\Documents\\file.ext
9. If user asks a question about screenshot: Answer directly in message, actions can be empty

PATH SHORTCUTS:
- documents → C:\\Users\\%USERNAME%\\Documents
- desktop → C:\\Users\\%USERNAME%\\Desktop
- downloads → C:\\Users\\%USERNAME%\\Downloads
- pictures → C:\\Users\\%USERNAME%\\Pictures

FILE CREATION WORKFLOW (applies to ALL file types):
Step 1: Create content (PowerPoint/Word/Excel/Paint/Publisher/Whiteboard/OneNote)
Step 2: Ask user where to save with requires_choice: true
Step 3: Wait for user response
Step 4: Save based on user choice OR discard if user says no

SMART DEFAULTS:
- PowerPoint → Desktop\\Topic_Presentation.pptx
- Word → Desktop\\Topic_Document.docx
- Excel → Desktop\\Topic_Spreadsheet.xlsx
- Paint → Pictures\\Drawing.png
- Publisher → Documents\\Publication.pub
- Whiteboard → Documents\\Whiteboard.wbd
- OneNote → Create in OneNote (no file save needed)
- Notepad → Desktop\\Notes.txt

CONVERSATIONAL EXAMPLES:

User: "create excel sheet with sales data"
{{
  "message": "I'll create an Excel spreadsheet with sales data columns (Date, Product, Quantity, Revenue). Once ready, I'll ask where you'd like to save it.",
  "actions": [
    {{"name": "create_excel_spreadsheet", "parameters": {{"content": "sales data with columns"}}}}
  ],
  "requires_choice": true,
  "choice_prompt": "✅ Your sales spreadsheet is ready! Where would you like to save it?\\n\\n• Type 'desktop' to save on Desktop\\n• Type 'documents' to save in Documents\\n• Or specify a custom path\\n\\nSuggested filename: Sales_Data.xlsx"
}}

User: "create word doc about meeting notes"
{{
  "message": "I'll create a Word document for your meeting notes with sections for attendees, agenda, discussion points, and action items. Once ready, I'll ask where to save it.",
  "actions": [
    {{"name": "create_word_document", "parameters": {{"content": "meeting notes template"}}}}
  ],
  "requires_choice": true,
  "choice_prompt": "✅ Your meeting notes document is ready! Where would you like to save it?\\n\\n• Type 'desktop' or 'documents'\\n• Or specify a custom path\\n\\nSuggested filename: Meeting_Notes.docx"
}}

User: "open paint and draw a circle"
{{
  "message": "Opening Paint and drawing a circle for you. Once you're done, I can help you save it.",
  "actions": [
    {{"name": "open_app", "parameters": {{"app_name": "mspaint"}}}},
    {{"name": "wait", "parameters": {{"ms": 2000}}}},
    {{"name": "draw_circle", "parameters": {{"x": 400, "y": 300, "radius": 100}}}}
  ],
  "requires_choice": true,
  "choice_prompt": "Would you like to save this drawing? Type 'yes' to save (I'll suggest Pictures folder) or 'no' to discard."
}}

User responds: "save it"
{{
  "message": "Saving your drawing to Pictures as Drawing.png.",
  "actions": [
    {{"name": "save_paint_image", "parameters": {{"location": "pictures", "filename": "Drawing.png"}}}}
  ],
  "requires_choice": false
}}

User: "create a 3 slide ppt about AI"
{{
  "message": "I'll create a 3-slide PowerPoint presentation about Artificial Intelligence for you. This will include professional layouts, themes, and content. Once it's ready, I'll ask you where you'd like to save it.",
  "actions": [
    {{"name": "create_presentation", "parameters": {{"topic": "Artificial Intelligence", "num_slides": 3}}}}
  ],
  "requires_choice": true,
  "choice_prompt": "✅ Your 3-slide AI presentation is ready! Where would you like to save it?\\n\\n• Type 'desktop' to save on Desktop\\n• Type 'documents' to save in Documents\\n• Or specify a custom path\\n\\nSuggested filename: AI_Presentation.pptx"
}}

User responds: "save on desktop as AI_Overview"
AI checks: AI_Overview.pptx already exists on Desktop
{{
  "message": "I found an existing file named AI_Overview.pptx on your Desktop. What would you like to do?",
  "actions": [],
  "requires_choice": true,
  "choice_prompt": "⚠️ AI_Overview.pptx already exists on Desktop!\\n\\nOptions:\\n• Type 'overwrite' to replace the existing file\\n• Type 'rename' to save as AI_Overview_2.pptx\\n• Type 'cancel' to discard the new presentation"
}}

User responds: "overwrite"
{{
  "message": "Understood! Overwriting the existing file with your new presentation and opening it.",
  "actions": [
    {{"name": "save_presentation", "parameters": {{"location": "desktop", "filename": "AI_Overview.pptx", "overwrite": true}}}}
  ],
  "requires_choice": false
}}

User responds: "no" or "cancel"
{{
  "message": "No problem! I've discarded the presentation. Let me know if you need anything else.",
  "actions": [],
  "requires_choice": false
}}

User: "set volume to 75"
{{
  "message": "Setting your system volume to 75%.",
  "actions": [
    {{"name": "set_volume", "parameters": {{"volume": 75}}}}
  ],
  "requires_choice": false
}}

FILE CONFLICT HANDLING (ALL FILE TYPES):
1. Check if file exists at target location
2. If exists: Pause and ask user (overwrite/rename/cancel) with requires_choice: true
3. Suggest auto-numbered filename (e.g., filename_2.ext)
4. Wait for user decision before saving

RESPOND WITH ONLY JSON - NO OTHER TEXT."""

        # Build prompt with LIMITED history (max 3 messages to save tokens)
        recent_history = conversation_history[-3:] if len(conversation_history) > 3 else conversation_history
        full_prompt = f"{system_prompt}\n\nUser request: {user_message}"
        
        # Tokenize
        inputs = text_tokenizer(full_prompt, return_tensors="pt").to(text_model.device)
        input_len = inputs.input_ids.shape[1]
        
        print(f"📊 Input length: {input_len} tokens")
        
        # CRITICAL: If input too large, truncate it
        MAX_INPUT_TOKENS = 4000
        if input_len > MAX_INPUT_TOKENS:
            print(f"⚠️ Input too large ({input_len} tokens), truncating to {MAX_INPUT_TOKENS}")
            inputs.input_ids = inputs.input_ids[:, -MAX_INPUT_TOKENS:]
            inputs.attention_mask = inputs.attention_mask[:, -MAX_INPUT_TOKENS:]
            input_len = MAX_INPUT_TOKENS
        
        # Generate with REDUCED max_new_tokens to prevent OOM
        with torch.no_grad():
            outputs = text_model.generate(
                inputs.input_ids,
                attention_mask=inputs.attention_mask,
                max_new_tokens=1500,  # REDUCED from 3000 to save memory
                temperature=0.7,
                do_sample=True,
                pad_token_id=text_tokenizer.eos_token_id,
                eos_token_id=text_tokenizer.eos_token_id
            )
        
        # Clear cache after generation
        torch.cuda.empty_cache()
        
        # Decode response
        response = text_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        
        print(f"\n=== MODEL RESPONSE ===")
        print(response)
        print(f"=== END RESPONSE ===\n")
        
        # Extract JSON from response
        actions = []
        message = ""
        expected_result = ""
        requires_choice = False
        choice_prompt = ""
        
        # Find JSON object by counting braces
        def extract_json_object(text):
            """Extract complete JSON object handling nested braces"""
            import re
            
            # Remove any markdown code blocks
            code_block_match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
            if code_block_match:
                text = code_block_match.group(1)
            
            # Remove common prefixes the model might add
            text = re.sub(r'^["\s]*(?:json|JSON)?\s*["\s]*', '', text)
            
            # Find the first opening brace
            start_idx = text.find('{')
            if start_idx == -1:
                print("⚠️ No opening brace found in response")
                return None
            
            # Start from the opening brace
            text = text[start_idx:]
            
            brace_count = 0
            in_string = False
            escape_next = False
            
            for i in range(len(text)):
                char = text[i]
                
                if escape_next:
                    escape_next = False
                    continue
                    
                if char == '\\':
                    escape_next = True
                    continue
                    
                if char == '"' and not escape_next:
                    in_string = not in_string
                    continue
                    
                if not in_string:
                    if char == '{':
                        brace_count += 1
                    elif char == '}':
                        brace_count -= 1
                        if brace_count == 0:
                            return text[:i+1]
            
            # If we didn't find closing brace, JSON is incomplete
            print(f"⚠️ Incomplete JSON - missing {brace_count} closing brace(s)")
            print(f"⚠️ Response ended at: ...{text[-100:]}")
            return None
        
        json_str = extract_json_object(response)
        
        # If no valid JSON found, try to salvage what we can
        if not json_str and '{' in response:
            print("⚠️ Attempting to repair incomplete JSON...")
            # Try to extract at least the message and actions that were generated
            start = response.find('{')
            partial = response[start:]
            # Count how many closing braces we need
            open_count = partial.count('{')
            close_count = partial.count('}')
            missing = open_count - close_count
            if missing > 0:
                # Add missing closing braces
                json_str = partial + ('}' * missing)
                print(f"⚠️ Added {missing} closing braces to complete JSON")
        
        if json_str:
            try:
                # Fix common issue: escape single backslashes in Windows paths
                json_str = re.sub(r'\\(?!["\\/bfnrtu])', r'\\\\', json_str)
                
                parsed = json.loads(json_str)
                actions = parsed.get('actions', [])
                message = parsed.get('message', '')
                expected_result = parsed.get('expected_result', '')
                requires_choice = parsed.get('requires_choice', False)
                choice_prompt = parsed.get('choice_prompt', '')
                print(f"✅ Extracted complete response with {len(actions)} actions")
            except Exception as e:
                print(f"❌ JSON parse error: {e}")
                print(f"Attempted to parse: {json_str[:500]}...")
                print(f"Full response length: {len(response)} chars")
        else:
            print(f"❌ No JSON object found in response")
            print(f"Response preview: {response[:500]}...")
            print(f"Response length: {len(response)} chars")
        
        return {
            "actions": actions,
            "message": message or response[:200],
            "expected_result": expected_result,
            "requires_choice": requires_choice,
            "choice_prompt": choice_prompt
        }
    except Exception as e:
        error_msg = f"Error: {str(e)}"
        print(f"❌ {error_msg}")
        import traceback
        traceback.print_exc()
        return {"actions": [], "message": error_msg, "requires_choice": False, "error": str(e)}


print("✅ Functions ready!")

## 8️⃣ Create API Server

In [ ]:
"""
8  Create API Server using langchain_backend.py
====================================================
langchain_backend.py handles ALL providers automatically.
The vision_model / text_model loaded above are detected automatically
when LLM_PROVIDER=transformers.
"""

import sys, os, nest_asyncio
import threading, time

nest_asyncio.apply()

# -- Copy backend file to /kaggle/working so it's importable --
import shutil
backend_src = "/kaggle/input/pecifics-backend/langchain_backend.py"
backend_dst = "/kaggle/working/langchain_backend.py"

# Try to find the file
if os.path.exists(backend_src):
    shutil.copy(backend_src, backend_dst)
    print(f"Copied backend from dataset")
elif not os.path.exists(backend_dst):
    print("langchain_backend.py not found in dataset - using inline server")

# -- If transformers provider: expose loaded models to backend --
if LLM_PROVIDER == "transformers":
    pass  # models are in this notebook's global scope

# -- Import and start FastAPI app --
if os.path.exists(backend_dst):
    sys.path.insert(0, "/kaggle/working")
    import importlib
    if "langchain_backend" in sys.modules:
        lb = importlib.reload(sys.modules["langchain_backend"])
    else:
        import langchain_backend as lb

    if LLM_PROVIDER == "transformers":
        import types
        g = vars(lb)
        if "text_model" in dir():
            lb.text_model     = text_model
            lb.text_tokenizer = text_tokenizer
        if "vision_model" in dir():
            lb.vision_model     = vision_model
            lb.vision_processor = vision_processor

    the_app = lb.app
    print(f"Backend loaded - provider: {LLM_PROVIDER}")
else:
    # Fallback: minimal inline server
    from fastapi import FastAPI
    from fastapi.middleware.cors import CORSMiddleware
    from fastapi.responses import JSONResponse
    from pydantic import BaseModel
    from typing import Optional, List, Dict
    import json, re

    the_app = FastAPI(title="Pecifics Kaggle Backend", version="2.0.0")
    the_app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

    class _Req(BaseModel):
        message: str
        screenshot: Optional[str] = None
        conversation_history: Optional[List[Dict]] = []
        screen_width: Optional[int] = 1920
        screen_height: Optional[int] = 1080

    @the_app.get("/health")
    async def _health():
        return {"status": "ok", "provider": "inline-transformers", 
                "vision": VISION_MODEL, "vision_type": VISION_TYPE}

    @the_app.post("/chat")
    async def _chat(req: _Req):
        screenshot_analysis = "No screenshot."
        if req.screenshot:
            try:
                screenshot_analysis = analyze_screenshot(req.screenshot, req.message, req.screen_width, req.screen_height)
            except Exception: pass
        return JSONResponse(content=generate_actions(req.message, screenshot_analysis, req.conversation_history or []))

    print("Inline fallback server ready (transformers only)")

# ═══════════════════════════════════════════════════════════════
# Add /vision_act endpoint for CogAgent (works with BOTH server modes)
# This is the endpoint that langchain_backend.py's call_vision_provider()
# calls when COGAGENT_URL is set to this Kaggle server's ngrok URL.
# ═══════════════════════════════════════════════════════════════
if VISION_TYPE == "cogagent":
    from pydantic import BaseModel as _BM
    from fastapi.responses import JSONResponse as _JR
    from typing import Optional as _Opt, List as _List

    class _VisionActReq(_BM):
        screenshot: str
        goal: str
        step_history: _List = []
        screen_width: _Opt[int] = 1920
        screen_height: _Opt[int] = 1080
        cogagent_url: _Opt[str] = None  # ignored here, we ARE the cogagent

    @the_app.post("/vision_act")
    async def _vision_act(req: _VisionActReq):
        """CogAgent vision agent: screenshot -> grounded action."""
        try:
            result = cogagent_vision_act(
                screenshot_b64=req.screenshot,
                goal=req.goal,
                step_history=req.step_history,
                screen_width=req.screen_width or 1920,
                screen_height=req.screen_height or 1080,
            )
            return _JR(content=result)
        except Exception as e:
            import traceback
            traceback.print_exc()
            return _JR(content={"action": "fail", "description": str(e)}, status_code=500)

    @the_app.get("/health")
    async def _health_cogagent():
        """Override /health to report CogAgent status."""
        gpu_mem = "unknown"
        try:
            import torch
            if torch.cuda.is_available():
                used = torch.cuda.memory_allocated() / 1024**3
                total = torch.cuda.get_device_properties(0).total_memory / 1024**3
                gpu_mem = f"{used:.1f}/{total:.1f}GB"
        except: pass
        return {
            "status": "ok", 
            "vision": "cogagent-9b-20241220",
            "vision_type": VISION_TYPE,
            "provider": LLM_PROVIDER,
            "gpu_memory": gpu_mem,
        }
    
    print("  /vision_act endpoint added (CogAgent-9B)")

import uvicorn

def _run():
    uvicorn.run(the_app, host="0.0.0.0", port=8000, log_level="warning")

server_thread = threading.Thread(target=_run, daemon=True)
server_thread.start()
time.sleep(3)
print("Server started on port 8000")

## 🔟 Start Server

In [ ]:
from pyngrok import ngrok as _ngrok

# Open tunnel
public_url = _ngrok.connect(8000)

print()
print("=" * 65)
print("🚀  Pecifics Backend is LIVE")
print("=" * 65)
print(f"   📡 Public URL  : {public_url}")
print(f"   🧠 Provider    : {LLM_PROVIDER}")
if LLM_PROVIDER == "transformers":
    print(f"   👁  Vision     : {VISION_MODEL}")
    print(f"   📝 Text model  : {TEXT_MODEL}")
elif LLM_PROVIDER == "groq":
    print(f"   🔥 Model       : {GROQ_MODEL}  (free Groq cloud)")
elif LLM_PROVIDER == "huggingface":
    print(f"   🤗 Model       : {HF_MODEL}  (HF Inference API)")
elif LLM_PROVIDER == "ollama":
    print(f"   🦙 Model       : {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}")
print()
print("   👉 Copy the URL above → paste into Pecifics Settings → Save")
print("=" * 65)

# Store for keep-alive cell
os.environ["PUBLIC_URL"] = str(public_url)


## ⚠️ Keep Running

In [ ]:
import psutil
from IPython.display import clear_output
import time

counter = 0
while True:
    counter += 1
    clear_output(wait=True)
    ram_avail = psutil.virtual_memory().available / (1024**3)
    print("🟢  Pecifics Backend running…")
    print(f"   📡 URL      : {public_url}")
    print(f"   🧠 Provider : {LLM_PROVIDER}")
    print(f"   ⏱️  Uptime   : {counter} min")
    print(f"   💾 RAM free : {ram_avail:.1f} GB")
    if ram_avail < 1.5:
        print("   ⚠️  LOW MEMORY WARNING")
    print()
    print("   Leave this cell running — it keeps the session alive.")
    time.sleep(60)
